In [35]:
from typing import Any

import altair as alt
import pandas as pd

from pathlib import Path

from analysis.models.openhands import Evaluation

# Altair stores the input data in all visualizations, and we're not being careful about the size of the data we're passing.
# If you want to export the visualizations and embed on the web, you might want to comment out this line and look into:
# https://altair-viz.github.io/user_guide/large_datasets.html#vegafusion-data-transformer
alt.data_transformers.disable_max_rows()

# Plug in filepaths to OpenHands evaluation data here -- anything produced using the OpenHands SWE-bench evaluation framework
# should be compatible.
filepaths = [
    Path("/Users/calvin/all-hands/data/condenser-blog/baseline-run-1"),
    Path("/Users/calvin/all-hands/data/condenser-blog/baseline-run-2"),
    Path("/Users/calvin/all-hands/data/condenser-blog/baseline-run-3"),
    Path("/Users/calvin/all-hands/data/condenser-blog/condenser-run-2"),
    Path("/Users/calvin/all-hands/data/condenser-blog/condenser-run-3"),
    Path("/Users/calvin/all-hands/data/condenser-blog/condenser-run-4"),
    Path("/Users/calvin/all-hands/data/cache-reuse/cache-reuse-run-1"),
    Path("/Users/calvin/all-hands/data/cache-reuse/cache-reuse-run-2"),
    Path("/Users/calvin/all-hands/data/cache-reuse/cache-reuse-run-3"),
]

data = [Evaluation.from_filepath(str(filepath)) for filepath in filepaths]

SHOULD_SAVE_CHARTS = False

In [36]:
from collections.abc import Iterable
from analysis.models.openhands import EvaluationOutput, SWEBenchResult
from analysis.usage import per_iteration_resource_usage


def per_step(output: EvaluationOutput, result: SWEBenchResult) -> Iterable[dict[str, Any]]:
    for step, step_usage in enumerate(per_iteration_resource_usage(output)):
        yield {
            "resolved": result.test_result.report.resolved,
            **step_usage.model_dump(),
            "iteration": step / 2,
        }

def post_process(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(by="iteration")
    df["cumulative_tokens"] = df["prompt_tokens"].cumsum()
    return df

df = pd.concat([d.multi_to_dataframe(per_step, post_callback=post_process) for d in data])
df = df[df['iteration'] % 1 == 0]
df["iteration"] = df["iteration"].astype(int)
df.reset_index(drop=True, inplace=True)

df['strategy'] = df['experiment'].map(lambda x: x.split('-')[0])
df['run'] = df['experiment'].map(lambda x: x.split('-')[-1])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21424 entries, 0 to 21423
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   experiment         21424 non-null  object 
 1   instance_id        21424 non-null  object 
 2   resolved           21424 non-null  bool   
 3   prompt_tokens      21424 non-null  int64  
 4   completion_tokens  21424 non-null  int64  
 5   cache_reads        21424 non-null  int64  
 6   cache_writes       21424 non-null  int64  
 7   response_latency   21424 non-null  float64
 8   accumulated_cost   21424 non-null  float64
 9   iteration          21424 non-null  int64  
 10  cumulative_tokens  21424 non-null  int64  
 11  strategy           21424 non-null  object 
 12  run                21424 non-null  object 
dtypes: bool(1), float64(2), int64(6), object(4)
memory usage: 2.0+ MB


In [37]:
# These runs have some bogus cost entries, so we'll need to recompute `cost` and `accumulated_cost`.

import litellm

def cost(row: pd.Series) -> float:
    input_costs, output_costs = litellm.cost_calculator.cost_per_token(
        model="claude-3-7-sonnet-20250219",
        prompt_tokens=row["prompt_tokens"],
        completion_tokens=row["completion_tokens"],
        cache_creation_input_tokens=row["cache_writes"],
        cache_read_input_tokens=row["cache_reads"],
    )
    return input_costs + output_costs

df["cost"] = df.apply(cost, axis=1)
df['accumulated_cost'] = df.sort_values(["experiment", "instance_id", "iteration"]).groupby(['experiment', 'instance_id'])['cost'].cumsum()

In [38]:
# I want bar graphs that show resource consumption at fun breakpoints.
# E.g., how many instances resolved on by the 50th iteration, 1 second of response latency, 1 dollar of cost, and 100,000 tokens used.
from functools import reduce


breakpoints = {
    "iteration": 50,
    "response_latency": 750,
    "cost": 0.75,
    "prompt_tokens": 1_000_000,
}

breakpoint_labels = {
    "iteration": "50 turns",
    "response_latency": "750ms",
    "cost": "$1.00",
    "prompt_tokens": "1M tokens",
}

resource_df = (
    df.groupby(["experiment", "instance_id"])
    .agg(
        {
            "strategy": "first",
            "resolved": "max",
            "iteration": "max",
            "cost": "sum",
            "prompt_tokens": "sum",
            "response_latency": "sum",
        }
    )
    .reset_index()
)


def breakpoint_comparison(
    df: pd.DataFrame, column: str, breakpoint: float
) -> pd.DataFrame:
    graph_df = df.copy()
    graph_df[f"{column}_breakpoint"] = graph_df[column].apply(
        lambda x: 1 if x <= breakpoint else 0
    )
    graph_df = (
        graph_df[graph_df["resolved"] == 1]
        .groupby("experiment")
        .agg(
            {
                f"{column}_breakpoint": "sum",
                "strategy": "first",
            }
        )
        .reset_index()
        .groupby("strategy")
        .agg(
            {
                f"{column}_breakpoint": "mean",
            }
        )
        .reset_index()
    )

    graph_df[f"{column}_breakpoint"] = graph_df[f"{column}_breakpoint"] / 50

    return graph_df


bar_df = reduce(
    lambda left, right: pd.merge(left, right, on="strategy", how="inner"),
    [
        breakpoint_comparison(resource_df, column, breakpoint)
        for column, breakpoint in breakpoints.items()
    ],
).melt(
    id_vars=["strategy"],
    value_vars=[f"{column}_breakpoint" for column in breakpoints.keys()],
    var_name="breakpoint",
    value_name="value",
)

bar_df["strategy"] = bar_df["strategy"].map(
    lambda x: {"condenser": "Condenser", "baseline": "Baseline", "cache": "Cache"}[x]
)


def rename_breakpoint(breakpoint: str) -> str:
    # split the string on the underscore
    column = "_".join(breakpoint.split("_")[:-1])
    return breakpoint_labels[column]


bar_df["breakpoint"] = bar_df["breakpoint"].apply(rename_breakpoint)

chart = alt.Chart(bar_df).mark_bar(size=30).encode(
    x=alt.X("strategy:N", title=None, axis=None, scale=alt.Scale(padding=2)),
    y=alt.Y(
        "value",
        title=None,
        axis=alt.Axis(ticks=False, format="%"),
        scale=alt.Scale(domain=[0.25, 0.55], clamp=True),
    ),
    color=alt.Color(
        "strategy:N",
        title="Strategy",
        scale=alt.Scale(domain=["Baseline", "Condenser", "Cache"], range=["#000000", "#FAE279", "#FF7F0E"]),
    ),
).facet(
    column=alt.Column(
        "breakpoint",
        title="Percent Total Problems Solved In Under...",
        header=alt.Header(
            titleFontSize=12,
            labelOrient="bottom",
            labelFontStyle="bold",
            labelFontSize=12,
        ),
    )
).configure_facet(spacing=0).resolve_scale(x="shared").configure_view(
    stroke=None,
    fill=None,
    discreteWidth=(300 * 16 / 9) / len(breakpoints),
    discreteHeight=300,
).configure_axis(domain=False)

if SHOULD_SAVE_CHARTS:
    chart.save("resource-usage.png", transparent=True, scale_factor=2)

chart

alt.FacetChart(...)

In [39]:
import statsmodels.api as sm
import numpy as np

graph_df = df.groupby(['experiment', 'iteration']).agg({
    "accumulated_cost": "mean",
    "cost": "mean",
    "resolved": "max",
    "strategy": "first"
}).reset_index()

# Apply LOWESS smoothing by experiment
def lowess_smooth(group):
    x = group['iteration'].values
    y = group['cost'].values
    # frac parameter controls smoothing (0.2 to 0.3 is often good)
    smoothed = sm.nonparametric.lowess(y, x, frac=0.1)
    return pd.Series(smoothed[:, 1], index=group.index)

graph_df["cost"] = graph_df.sort_values(['experiment', 'iteration']).groupby('experiment').apply(
    lambda x: lowess_smooth(x)
).reset_index(level=0, drop=True)

graph_df["strategy"] = graph_df["strategy"].map(
    lambda x: {"condenser": "Condenser", "baseline": "Baseline", "cache": "Cache"}[x]
)

chart = alt.Chart(graph_df, title="API Cost Per-Turn").mark_line(size=5).encode(
    x=alt.X("iteration",
        title="Turns",
        scale=alt.Scale(domain=[0, 150]),
        axis=alt.Axis(
            values=[0, 25, 50, 75, 100, 125, 150],
        )
    ),
    y=alt.Y(
        "mean(cost)",
        title=None,
        # scale=alt.Scale(domain=[0.01, 0.05]),
        axis=alt.Axis(
            format=".0f",
            labelExpr="(datum.value * 100) + '¢'",
            # values=[0.01, 0.02, 0.03, 0.04, 0.05],
        ),
    ),
    color=alt.Color(
        "strategy:N",
        title="Strategy",
        scale=alt.Scale(domain=["Baseline", "Condenser", "Cache"], range=["#000000", "#FAE279", "#FF7F0E"]),
    ),
).configure_legend(
    symbolStrokeWidth=5,
).configure_view(
    fill=None,
    continuousWidth=300 * 16 / 9,
    continuousHeight=300,
)

if SHOULD_SAVE_CHARTS:
    chart.save("cost-per-turn.png", transparent=True, scale_factor=2)

chart

/var/folders/gp/5273pn213bq0jn9l40pvspx80000gn/T/ipykernel_78684/1007398823.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  graph_df["cost"] = graph_df.sort_values(['experiment', 'iteration']).groupby('experiment').apply(


alt.Chart(...)

In [47]:
agg = df.groupby(["experiment", "instance_id"]).agg(
    {
        "resolved": "max",
        "strategy": "first",
        "accumulated_cost": "max",
        "iteration": "max",
    }
).reset_index().groupby("experiment").agg({
    "resolved": "mean",
    "strategy": "first",
    "accumulated_cost": "mean",
    "iteration": "mean",
}).reset_index().groupby("strategy").agg({
    "resolved": "mean",
    "accumulated_cost": "mean",
    "iteration": "mean",
})

agg["avg_cost_per_iteration"] = agg["accumulated_cost"] / agg["iteration"]
agg

,resolved,accumulated_cost,iteration,avg_cost_per_iteration
strategy,,,,
baseline,0.526667,1.219408,49.466667,0.024651
cache,0.473333,0.698510,34.533333,0.020227
condenser,0.540000,1.211745,55.826667,0.021705


In [46]:
agg = df.groupby(["experiment", "instance_id"]).agg(
    {
        "resolved": "max",
        "strategy": "first",
        "iteration": "max"
    }
).reset_index()

agg = agg[agg["resolved"] == 1]
agg = agg[agg["iteration"] > 40]

agg

,experiment,instance_id,resolved,strategy,iteration
0,baseline-run-1,astropy__astropy-12907,True,baseline,133
11,baseline-run-1,django__django-12262,True,baseline,97
15,baseline-run-1,django__django-13158,True,baseline,74
17,baseline-run-1,django__django-13670,True,baseline,47
20,baseline-run-1,django__django-15278,True,baseline,50
22,baseline-run-1,django__django-15916,True,baseline,147
31,baseline-run-1,scikit-learn__scikit-learn-14983,True,baseline,48
42,baseline-run-1,sympy__sympy-15599,True,baseline,58
47,baseline-run-1,sympy__sympy-23534,True,baseline,60
65,baseline-run-2,django__django-13158,True,baseline,48
